# 🥛 Data Preprocessing — Milk Quality Prediction
**Dataset:** milknew.csv (Kaggle: cpluzshrijayan/milkquality)  
**Metode:** K-Means Clustering + Logistic Regression + Naïve Bayes  
**Tahap:** Data Preprocessing


In [ ]:
# ── Import Library ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler

import warnings
warnings.filterwarnings('ignore')

# Style plot
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print("✅ Semua library berhasil diimport!")


## 1. Load Dataset

In [ ]:
df = pd.read_csv('milknew.csv')

print(f"Shape  : {df.shape[0]} baris × {df.shape[1]} kolom")
print(f"Kolom  : {list(df.columns)}")
print()
df.head(10)


In [ ]:
print("=== Info DataFrame ===")
df.info()


In [ ]:
print("=== Statistik Deskriptif ===")
df.describe().round(3)


## 2. Pemeriksaan Missing Values

In [ ]:
missing = df.isnull().sum().reset_index()
missing.columns = ['Kolom', 'Jumlah Missing']
missing['Persentase (%)'] = (missing['Jumlah Missing'] / len(df) * 100).round(2)
print(missing.to_string(index=False))

# Visualisasi
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2ecc71' if v == 0 else '#e74c3c' for v in missing['Jumlah Missing']]
bars = ax.bar(missing['Kolom'], missing['Jumlah Missing'], color=colors, edgecolor='white', linewidth=1.2)
ax.set_title('Jumlah Missing Values per Kolom', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Kolom')
ax.set_ylabel('Jumlah Missing')
ax.set_ylim(0, max(missing['Jumlah Missing'].max() + 5, 10))
for bar, val in zip(bars, missing['Jumlah Missing']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(val), ha='center', va='bottom', fontweight='bold', fontsize=10)
patch_ok  = mpatches.Patch(color='#2ecc71', label='Tidak ada missing')
patch_err = mpatches.Patch(color='#e74c3c', label='Ada missing')
ax.legend(handles=[patch_ok, patch_err], loc='upper right')
plt.tight_layout()
plt.savefig('plot_missing_values.png', bbox_inches='tight')
plt.show()
print("✅ Tidak ditemukan missing value.")


## 3. Pemeriksaan & Penghapusan Duplikasi

In [ ]:
n_before  = len(df)
n_dup     = df.duplicated().sum()
n_after   = n_before - n_dup

print(f"Jumlah baris sebelum  : {n_before}")
print(f"Jumlah baris duplikat : {n_dup}")
print(f"Jumlah baris sesudah  : {n_after}")

# Pie chart
fig, ax = plt.subplots(figsize=(6, 6))
sizes  = [n_after, n_dup]
labels = [f'Unik\n({n_after} baris)', f'Duplikat\n({n_dup} baris)']
colors = ['#3498db', '#e74c3c']
explode = (0.05, 0.05)
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors, explode=explode,
    autopct='%1.1f%%', startangle=140,
    textprops={'fontsize': 11}, pctdistance=0.75)
for at in autotexts:
    at.set_fontweight('bold')
ax.set_title('Proporsi Data Unik vs Duplikat', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('plot_duplikasi.png', bbox_inches='tight')
plt.show()

# Drop duplikat
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f"\n✅ Duplikat berhasil dihapus. Shape sekarang: {df_clean.shape}")


## 4. Distribusi Kelas (Grade)

In [ ]:
grade_before = df['Grade'].value_counts().reset_index()
grade_before.columns = ['Grade', 'Sebelum Dedup']
grade_after  = df_clean['Grade'].value_counts().reset_index()
grade_after.columns  = ['Grade', 'Sesudah Dedup']
dist = grade_before.merge(grade_after, on='Grade')
dist['Pct Sebelum'] = (dist['Sebelum Dedup'] / dist['Sebelum Dedup'].sum() * 100).round(1)
dist['Pct Sesudah'] = (dist['Sesudah Dedup'] / dist['Sesudah Dedup'].sum() * 100).round(1)
print(dist.to_string(index=False))

# Side-by-side bar
x     = np.arange(len(dist))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - width/2, dist['Sebelum Dedup'], width, label='Sebelum Dedup',
            color='#3498db', edgecolor='white')
b2 = ax.bar(x + width/2, dist['Sesudah Dedup'], width, label='Sesudah Dedup',
            color='#2ecc71', edgecolor='white')
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(int(bar.get_height())), ha='center', va='bottom',
            fontsize=10, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(dist['Grade'], fontsize=12)
ax.set_xlabel('Grade Kualitas Susu')
ax.set_ylabel('Jumlah Data')
ax.set_title('Distribusi Kelas Sebelum & Sesudah Deduplikasi', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('plot_distribusi_grade.png', bbox_inches='tight')
plt.show()


## 5. Deteksi Outlier (Metode IQR)

In [ ]:
# Rename kolom 'Fat ' (ada trailing space)
df_clean = df_clean.rename(columns={'Fat ': 'Fat'})

numeric_cols = ['pH', 'Temprature', 'Colour']

# Tabel ringkasan outlier
rows = []
for col in numeric_cols:
    Q1  = df_clean[col].quantile(0.25)
    Q3  = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lb  = Q1 - 1.5 * IQR
    ub  = Q3 + 1.5 * IQR
    n_out = ((df_clean[col] < lb) | (df_clean[col] > ub)).sum()
    rows.append({'Kolom': col, 'Q1': Q1, 'Q3': Q3, 'IQR': round(IQR,2),
                 'Lower Bound': round(lb,2), 'Upper Bound': round(ub,2),
                 'Jumlah Outlier': n_out, 'Keputusan': 'Dipertahankan ✅'})
print(pd.DataFrame(rows).to_string(index=False))

# Boxplot
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(13, 5))
palette = ['#9b59b6', '#e67e22', '#1abc9c']
for ax, col, color in zip(axes, numeric_cols, palette):
    data_by_grade = [df_clean[df_clean['Grade']==g][col].values for g in ['low','medium','high']]
    bp = ax.boxplot(data_by_grade, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2))
    for patch, c in zip(bp['boxes'], ['#e74c3c','#f39c12','#2ecc71']):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)
    ax.set_xticklabels(['Low','Medium','High'])
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel('Grade')
    ax.set_ylabel('Nilai')
fig.suptitle('Boxplot Outlier per Grade (Kolom Numerik Kontinu)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_boxplot_outlier.png', bbox_inches='tight')
plt.show()
print("\n📌 Outlier DIPERTAHANKAN — nilai ekstrem pH/Temprature berkaitan dengan susu kualitas rendah (domain-valid).")


## 6. Encoding Label Target (Grade)

In [ ]:
le = LabelEncoder()
df_clean['Grade_encoded'] = le.fit_transform(df_clean['Grade'])

mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Mapping Label Encoding:")
for k, v in mapping.items():
    print(f"  {k:8s} → {v}")
print()
print(df_clean[['Grade','Grade_encoded']].value_counts().reset_index().sort_values('Grade').to_string(index=False))

# Bar chart hasil encoding
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
grade_counts = df_clean['Grade'].value_counts()
grade_enc    = df_clean['Grade_encoded'].value_counts().sort_index()
colors_bar   = ['#e74c3c', '#f39c12', '#2ecc71']

axes[0].bar(grade_counts.index, grade_counts.values, color=colors_bar, edgecolor='white')
axes[0].set_title('Distribusi Grade (Label Asli)', fontweight='bold')
axes[0].set_xlabel('Grade'); axes[0].set_ylabel('Jumlah')
for i, (lbl, val) in enumerate(grade_counts.items()):
    axes[0].text(i, val+0.3, str(val), ha='center', fontweight='bold')

axes[1].bar([str(k) for k in grade_enc.index], grade_enc.values,
            color=colors_bar, edgecolor='white')
axes[1].set_title('Distribusi Grade (Label Encoded)', fontweight='bold')
axes[1].set_xlabel('Grade Encoded (0=high,1=low,2=medium)')
axes[1].set_ylabel('Jumlah')
for i, val in enumerate(grade_enc.values):
    axes[1].text(i, val+0.3, str(val), ha='center', fontweight='bold')

plt.suptitle('Perbandingan Label Asli vs Encoded', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_encoding.png', bbox_inches='tight')
plt.show()


## 7. Feature Scaling

In [ ]:
features = ['pH','Temprature','Taste','Odor','Fat','Turbidity','Colour']

# MinMax Scaler
scaler_mm  = MinMaxScaler()
df_minmax  = df_clean.copy()
df_minmax[features] = scaler_mm.fit_transform(df_clean[features])

# Standard Scaler
scaler_std  = StandardScaler()
df_standard = df_clean.copy()
df_standard[features] = scaler_std.fit_transform(df_clean[features])

print("=== Contoh data setelah MinMaxScaler (5 baris) ===")
print(df_minmax[features].head().round(4).to_string())
print()
print("=== Contoh data setelah StandardScaler (5 baris) ===")
print(df_standard[features].head().round(4).to_string())

# Visualisasi distribusi sebelum vs sesudah scaling (pilih 3 kolom)
viz_cols  = ['pH', 'Temprature', 'Colour']
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for i, col in enumerate(viz_cols):
    # Before
    axes[i][0].hist(df_clean[col], bins=15, color='#3498db', edgecolor='white', alpha=0.85)
    axes[i][0].set_title(f'{col} — Original', fontweight='bold')
    axes[i][0].set_ylabel('Frekuensi')
    # MinMax
    axes[i][1].hist(df_minmax[col], bins=15, color='#2ecc71', edgecolor='white', alpha=0.85)
    axes[i][1].set_title(f'{col} — MinMaxScaler', fontweight='bold')
    # Standard
    axes[i][2].hist(df_standard[col], bins=15, color='#e67e22', edgecolor='white', alpha=0.85)
    axes[i][2].set_title(f'{col} — StandardScaler', fontweight='bold')

fig.suptitle('Distribusi Fitur: Original vs MinMax vs Standard Scaling',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plot_scaling_distribusi.png', bbox_inches='tight')
plt.show()
print("\n✅ MinMaxScaler → untuk K-Means & Logistic Regression")
print("✅ StandardScaler → untuk Gaussian Naïve Bayes")


## 8. Analisis Korelasi Fitur

In [ ]:
corr_matrix = df_clean[features + ['Grade_encoded']].corr()

# Heatmap
fig, ax = plt.subplots(figsize=(10, 7))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = False
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            annot_kws={'size': 10})
ax.set_title('Heatmap Korelasi Antar Fitur & Target', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('plot_heatmap_korelasi.png', bbox_inches='tight')
plt.show()

# Bar korelasi vs target
corr_target = corr_matrix['Grade_encoded'].drop('Grade_encoded').sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
colors_corr = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr_target]
bars = ax.barh(corr_target.index, corr_target.values, color=colors_corr, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
for bar, val in zip(bars, corr_target.values):
    ax.text(val + (0.01 if val >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right',
            fontsize=10, fontweight='bold')
ax.set_title('Korelasi Fitur terhadap Grade (Target)', fontsize=13, fontweight='bold')
ax.set_xlabel('Nilai Korelasi')
plt.tight_layout()
plt.savefig('plot_korelasi_target.png', bbox_inches='tight')
plt.show()
print("\n📌 Fat, Odor, dan Turbidity adalah fitur paling berpengaruh terhadap kualitas susu.")


## 9. Simpan Hasil Preprocessing

In [ ]:
# Simpan semua versi dataset
df_clean.to_csv('milknew_clean.csv', index=False)
df_minmax.to_csv('milknew_minmax.csv', index=False)
df_standard.to_csv('milknew_standard.csv', index=False)

print("✅ File berhasil disimpan:")
print("   • milknew_clean.csv    — data bersih (tanpa scaling)")
print("   • milknew_minmax.csv   — MinMaxScaler (K-Means & Logistic Regression)")
print("   • milknew_standard.csv — StandardScaler (Naïve Bayes)")
print(f"\nShape final: {df_clean.shape}")
print(f"Kolom: {list(df_clean.columns)}")


## 10. Ringkasan Preprocessing

| Tahap | Aksi | Hasil |
|---|---|---|
| **Load Data** | pd.read_csv | 1.059 baris × 8 kolom |
| **Missing Values** | Pemeriksaan | ✅ Tidak ada missing value |
| **Duplikasi** | drop_duplicates() | 1.059 → **83 baris** |
| **Outlier** | IQR detection | Dipertahankan (domain-valid) |
| **Rename Kolom** | `Fat ` → `Fat` | Trailing space dihapus |
| **Label Encoding** | LabelEncoder pada Grade | high=0, low=1, medium=2 |
| **MinMax Scaling** | MinMaxScaler (0–1) | Untuk K-Means & Logistic Regression |
| **Standard Scaling** | StandardScaler (z-score) | Untuk Naïve Bayes |
| **Analisis Korelasi** | Pearson correlation | Fat & Odor paling signifikan |

> **Catatan:** 976 baris duplikat (92%) kemungkinan akibat cara pengumpulan dataset di Kaggle. Data setelah deduplikasi (83 baris) digunakan untuk pemodelan.


---
## 11. GridSearchCV — Optimasi Hyperparameter

GridSearchCV digunakan untuk mencari kombinasi hyperparameter terbaik secara exhaustive melalui cross-validation.

In [ ]:
# ── Import tambahan untuk modelling ────────────────────────────────────
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    silhouette_score, adjusted_rand_score, ConfusionMatrixDisplay
)
from sklearn.decomposition import PCA

print('✅ Library modelling berhasil diimport!')

In [ ]:
# ── Persiapan data dari hasil preprocessing ─────────────────────────────
# Gunakan df_minmax (MinMaxScaler) → untuk Logistic Regression & K-Means
# Gunakan df_standard (StandardScaler) → untuk Naïve Bayes

features = ['pH','Temprature','Taste','Odor','Fat','Turbidity','Colour']
target   = 'Grade_encoded'

X_mm  = df_minmax[features]
X_std = df_standard[features]
y     = df_clean[target]

# Split data (80:20, stratified)
X_train_mm,  X_test_mm,  y_train, y_test = train_test_split(
    X_mm,  y, test_size=0.2, random_state=42, stratify=y)
X_train_std, X_test_std, _,       _      = train_test_split(
    X_std, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train : {X_train_mm.shape[0]} sampel')
print(f'Test  : {X_test_mm.shape[0]}  sampel')
print(f'Kelas : {le.classes_}  →  {list(le.transform(le.classes_))}')

In [ ]:
# ── GridSearchCV — Logistic Regression ──────────────────────────────────
param_grid_lr = {
    'C'          : [0.01, 0.1, 1, 10, 100],
    'solver'     : ['lbfgs', 'saga'],
    'max_iter'   : [200, 500]
}

grid_lr = GridSearchCV(
    LogisticRegression(random_state=42, multi_class='auto'),
    param_grid_lr,
    cv=5, scoring='accuracy', n_jobs=-1, verbose=0
)
grid_lr.fit(X_train_mm, y_train)

print('=== GridSearchCV — Logistic Regression ===')
print(f'Best Params  : {grid_lr.best_params_}')
print(f'Best CV Acc  : {grid_lr.best_score_:.4f}')

best_lr = grid_lr.best_estimator_

In [ ]:
# ── Visualisasi hasil GridSearchCV (heatmap CV score) ───────────────────
import pandas as pd
results_lr = pd.DataFrame(grid_lr.cv_results_)
# Filter hanya solver lbfgs & max_iter=200 untuk heatmap 2D
pivot_lr = results_lr[
    (results_lr['param_solver'] == 'lbfgs') &
    (results_lr['param_max_iter'] == 200)
].pivot_table(
    values='mean_test_score',
    index='param_solver',
    columns='param_C'
)

fig, ax = plt.subplots(figsize=(9, 3))
sns.heatmap(pivot_lr, annot=True, fmt='.4f', cmap='YlGn',
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title('GridSearchCV CV Accuracy — Logistic Regression\n(solver=lbfgs, max_iter=200)',
             fontweight='bold')
ax.set_xlabel('Nilai C (Regularization)')
plt.tight_layout()
plt.savefig('plot_gridsearch_lr.png', bbox_inches='tight')
plt.show()

# Bar: semua kombinasi C vs mean test score
agg = results_lr.groupby('param_C')['mean_test_score'].mean().reset_index()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(agg['param_C'].astype(str), agg['mean_test_score'],
       color='#3498db', edgecolor='white')
ax.set_title('Rata-rata CV Accuracy per Nilai C (Logistic Regression)',
             fontweight='bold')
ax.set_xlabel('Nilai C'); ax.set_ylabel('Mean CV Accuracy')
ax.set_ylim(0, 1.05)
for bar, val in zip(ax.patches, agg['mean_test_score']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_gridsearch_C.png', bbox_inches='tight')
plt.show()

---
## 12. K-Means Clustering

K-Means adalah algoritma unsupervised learning yang mengelompokkan data ke dalam **k** cluster berdasarkan kemiripan fitur. Dataset susu memiliki 3 kelas (low/medium/high), sehingga k=3 menjadi pilihan awal yang logis. Metode Elbow dan Silhouette Score digunakan untuk validasi.

In [ ]:
# ── Elbow Method — cari k optimal ───────────────────────────────────────
inertia_list = []
sil_list     = []
k_range      = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_mm)
    inertia_list.append(km.inertia_)
    sil_list.append(silhouette_score(X_mm, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Elbow
axes[0].plot(list(k_range), inertia_list, marker='o', color='#3498db',
             linewidth=2, markersize=7)
axes[0].axvline(3, color='#e74c3c', linestyle='--', label='k=3 (optimal)')
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].set_xlabel('Jumlah Cluster (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].legend(); axes[0].grid(True, alpha=0.4)

# Silhouette
axes[1].plot(list(k_range), sil_list, marker='s', color='#2ecc71',
             linewidth=2, markersize=7)
axes[1].axvline(k_range[sil_list.index(max(sil_list))], color='#e74c3c',
                linestyle='--', label=f'k={k_range[sil_list.index(max(sil_list))]} (max silhouette)')
axes[1].set_title('Silhouette Score per k', fontweight='bold')
axes[1].set_xlabel('Jumlah Cluster (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend(); axes[1].grid(True, alpha=0.4)

plt.suptitle('Pemilihan k Optimal — K-Means Clustering', fontsize=13,
             fontweight='bold')
plt.tight_layout()
plt.savefig('plot_kmeans_elbow.png', bbox_inches='tight')
plt.show()

print(f'Inertia per k   : {dict(zip(k_range, [round(v,2) for v in inertia_list]))}')
print(f'Silhouette per k: {dict(zip(k_range, [round(v,4) for v in sil_list]))}')

In [ ]:
# ── GridSearchCV untuk KMeans (n_clusters & n_init) ─────────────────────
# Catatan: KMeans tidak support GridSearchCV langsung via scoring standar.
# Kita buat manual grid dengan Silhouette Score sebagai metrik.

from itertools import product as iterproduct

param_grid_km = {
    'n_clusters': [2, 3, 4, 5],
    'n_init'    : [5, 10, 15],
    'init'      : ['k-means++', 'random']
}

km_results = []
for nc, ni, init in iterproduct(
        param_grid_km['n_clusters'],
        param_grid_km['n_init'],
        param_grid_km['init']):
    km_tmp = KMeans(n_clusters=nc, n_init=ni, init=init, random_state=42)
    labels_tmp = km_tmp.fit_predict(X_mm)
    sil = silhouette_score(X_mm, labels_tmp)
    km_results.append({'n_clusters': nc, 'n_init': ni,
                        'init': init, 'silhouette': round(sil, 4)})

df_km_grid = pd.DataFrame(km_results).sort_values('silhouette', ascending=False)
print('=== Top 10 Konfigurasi KMeans (by Silhouette Score) ===')
print(df_km_grid.head(10).to_string(index=False))

best_km_params = df_km_grid.iloc[0]
print(f'\n✅ Best: n_clusters={int(best_km_params.n_clusters)}, '
      f'n_init={int(best_km_params.n_init)}, '
      f'init={best_km_params.init}, '
      f'silhouette={best_km_params.silhouette}')

In [ ]:
# ── Fit KMeans dengan k=3 (sesuai jumlah kelas susu) ────────────────────
best_k    = 3
best_init = best_km_params['init']
best_ni   = int(best_km_params['n_init'])

kmeans = KMeans(n_clusters=best_k, n_init=best_ni,
                init=best_init, random_state=42)
cluster_labels = kmeans.fit_predict(X_mm)

df_cluster = df_minmax.copy()
df_cluster['Cluster'] = cluster_labels

sil_final = silhouette_score(X_mm, cluster_labels)
ari_final = adjusted_rand_score(y, cluster_labels)

print(f'Silhouette Score : {sil_final:.4f}')
print(f'Adjusted Rand Index (vs Grade) : {ari_final:.4f}')
print()
print('Distribusi cluster:')
print(df_cluster['Cluster'].value_counts().sort_index().to_string())

In [ ]:
# ── Visualisasi Cluster (PCA 2D) ─────────────────────────────────────────
pca2 = PCA(n_components=2, random_state=42)
X_pca = pca2.fit_transform(X_mm)

centers_pca = pca2.transform(kmeans.cluster_centers_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cluster_colors = ['#e74c3c', '#3498db', '#2ecc71']
grade_colors   = ['#e74c3c', '#f39c12', '#2ecc71']

# Plot cluster hasil KMeans
for c in range(best_k):
    mask_c = cluster_labels == c
    axes[0].scatter(X_pca[mask_c, 0], X_pca[mask_c, 1],
                    c=cluster_colors[c], label=f'Cluster {c}',
                    alpha=0.7, s=50, edgecolors='white', linewidth=0.4)
axes[0].scatter(centers_pca[:, 0], centers_pca[:, 1],
                c='black', marker='X', s=180, label='Centroid', zorder=5)
axes[0].set_title('K-Means Clustering (PCA 2D)', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend()

# Plot label asli Grade
grade_vals = y.values
for g, gc in zip([0, 1, 2], grade_colors):
    mask_g = grade_vals == g
    lbl = le.inverse_transform([g])[0]
    axes[1].scatter(X_pca[mask_g, 0], X_pca[mask_g, 1],
                    c=gc, label=f'{lbl} (encoded={g})',
                    alpha=0.7, s=50, edgecolors='white', linewidth=0.4)
axes[1].set_title('Label Asli Grade (PCA 2D)', fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].legend()

plt.suptitle('Perbandingan Cluster K-Means vs Label Grade Asli (PCA 2D)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_kmeans_cluster.png', bbox_inches='tight')
plt.show()

print(f'\n📌 PCA menjelaskan {sum(pca2.explained_variance_ratio_)*100:.1f}% variansi total.')

In [ ]:
# ── Profil tiap cluster ──────────────────────────────────────────────────
cluster_profile = df_cluster.groupby('Cluster')[features].mean().round(3)
print('=== Rata-rata Fitur per Cluster (MinMax Scaled) ===')
print(cluster_profile.to_string())

# Heatmap profil cluster
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(cluster_profile, annot=True, fmt='.3f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title('Profil Rata-rata Fitur per Cluster', fontweight='bold')
ax.set_xlabel('Fitur'); ax.set_ylabel('Cluster')
plt.tight_layout()
plt.savefig('plot_kmeans_profil.png', bbox_inches='tight')
plt.show()

# Cross-tabulation cluster vs grade asli
cross_tab = pd.crosstab(cluster_labels, df_clean['Grade'],
                         rownames=['Cluster'], colnames=['Grade Asli'])
print('\n=== Cross-tabulation Cluster vs Grade Asli ===')
print(cross_tab.to_string())

---
## 13. Regresi — Logistic Regression

Logistic Regression digunakan untuk klasifikasi multikelas kualitas susu (low / medium / high). Model terbaik dari GridSearchCV (Section 11) digunakan di sini.

In [ ]:
# ── Evaluasi Logistic Regression (best params dari GridSearchCV) ─────────
y_pred_lr = best_lr.predict(X_test_mm)

acc_lr = accuracy_score(y_test, y_pred_lr)
cv_lr  = cross_val_score(best_lr, X_mm, y, cv=5, scoring='accuracy')

print('=== Logistic Regression (Best Params dari GridSearchCV) ===')
print(f'Best Params       : {grid_lr.best_params_}')
print(f'Test Accuracy     : {acc_lr:.4f}')
print(f'CV Accuracy (5-fold): {cv_lr.mean():.4f} ± {cv_lr.std():.4f}')
print()
print('=== Classification Report ===')
print(classification_report(y_test, y_pred_lr,
      target_names=le.classes_))

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────
cm_lr = confusion_matrix(y_test, y_pred_lr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr,
                               display_labels=le.classes_)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix — Logistic Regression', fontweight='bold')

# CV Score bar
fold_labels = [f'Fold {i+1}' for i in range(len(cv_lr))]
axes[1].bar(fold_labels, cv_lr, color='#3498db', edgecolor='white')
axes[1].axhline(cv_lr.mean(), color='#e74c3c', linestyle='--',
                label=f'Mean = {cv_lr.mean():.4f}')
axes[1].set_ylim(0, 1.1)
axes[1].set_title('5-Fold Cross Validation Accuracy', fontweight='bold')
axes[1].set_ylabel('Accuracy'); axes[1].legend()
for bar, val in zip(axes[1].patches, cv_lr):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Evaluasi Logistic Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_lr_eval.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Probabilitas prediksi & Decision Boundary (PCA 2D) ──────────────────
import numpy as np

pca2_lr = PCA(n_components=2, random_state=42)
X_pca_lr = pca2_lr.fit_transform(X_mm)

lr_pca = LogisticRegression(**grid_lr.best_params_, random_state=42,
                             multi_class='auto')
lr_pca.fit(X_pca_lr, y)

# Grid untuk decision boundary
h = 0.02
x_min, x_max = X_pca_lr[:,0].min()-0.3, X_pca_lr[:,0].max()+0.3
y_min, y_max = X_pca_lr[:,1].min()-0.3, X_pca_lr[:,1].max()+0.3
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                      np.arange(y_min, y_max, h))
Z = lr_pca.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(9, 6))
cmap_bg = plt.cm.get_cmap('Pastel1', 3)
ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_bg)
scatter_colors = ['#e74c3c', '#f39c12', '#2ecc71']
for g, sc in zip([0, 1, 2], scatter_colors):
    mask = y.values == g
    ax.scatter(X_pca_lr[mask,0], X_pca_lr[mask,1],
               c=sc, label=le.inverse_transform([g])[0],
               edgecolors='white', linewidth=0.5, s=60, alpha=0.9)
ax.set_title('Decision Boundary — Logistic Regression (PCA 2D)',
             fontweight='bold')
ax.set_xlabel(f'PC1 ({pca2_lr.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca2_lr.explained_variance_ratio_[1]*100:.1f}%)')
ax.legend(title='Grade')
plt.tight_layout()
plt.savefig('plot_lr_boundary.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Koefisien fitur Logistic Regression ──────────────────────────────────
coef_df = pd.DataFrame(
    best_lr.coef_,
    columns=features,
    index=[f'Class {le.inverse_transform([i])[0]}' for i in range(3)]
).T.round(4)

print('=== Koefisien Logistic Regression per Kelas ===')
print(coef_df.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
coef_df.plot(kind='bar', ax=ax, edgecolor='white', width=0.65)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Koefisien Fitur — Logistic Regression (per Kelas)',
             fontweight='bold')
ax.set_xlabel('Fitur'); ax.set_ylabel('Koefisien')
ax.set_xticklabels(features, rotation=30, ha='right')
ax.legend(title='Kelas')
plt.tight_layout()
plt.savefig('plot_lr_koefisien.png', bbox_inches='tight')
plt.show()

print('\n✅ Logistic Regression selesai dievaluasi.')

---
## 14. Ringkasan Modelling

| Tahap | Model / Metode | Metrik Utama |
|---|---|---|
| **GridSearchCV** | Logistic Regression | Best CV Accuracy |
| **Clustering** | K-Means (k=3) | Silhouette Score, ARI |
| **Klasifikasi** | Logistic Regression (best params) | Accuracy, F1-score |

> **Catatan:** Dataset setelah deduplikasi hanya memiliki 83 baris. Jumlah ini kecil, sehingga hasil evaluasi perlu diinterpretasikan dengan hati-hati. Pertimbangkan augmentasi data atau penggunaan dataset yang lebih besar untuk produksi.

In [ ]:
# ── Ringkasan metrik akhir ───────────────────────────────────────────────
summary = {
    'Model'         : ['K-Means (k=3)', 'Logistic Regression'],
    'Metrik'        : ['Silhouette Score', 'Test Accuracy'],
    'Nilai'         : [round(sil_final, 4), round(acc_lr, 4)],
    'CV / ARI'      : [round(ari_final, 4), f'{cv_lr.mean():.4f} ± {cv_lr.std():.4f}'],
    'Best Params'   : [f'n_init={best_ni}, init={best_init}',
                        str(grid_lr.best_params_)]
}
print(pd.DataFrame(summary).to_string(index=False))
print('\n✅ Semua tahap modelling selesai!')